# 실습 4. OpenAI API 챗봇 서비스 — 멀티턴 메모리 (Day 2 / 150분)

> 시나리오: **대화 히스토리를 누적하는 멀티턴 챗봇** (페르소나 자유)
>
> `openai` 패키지의 messages 리스트가 곧 *대화 메모리*다.
> (LangChain 의 ConversationBufferMemory 와 같은 개념을 직접 구현한다.)

## Task
1. 히스토리 챗봇 — 대화 누적 구조 (`/reset` 초기화)
2. 페르소나 변경 (system prompt) · temperature 0 vs 0.7
3. 환각 발견 — 모르는 사실 묻기 → system 으로 줄이기
4. 토큰/비용 모니터링 — 100턴 누적 비용 예측 + trim 전략

## 0) 준비

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-4o-mini"

SYSTEM = "당신은 친절한 AI 어시스턴트입니다. 모르면 '확인 필요'라고만 답하세요."

## 1) Task 1 — 히스토리 챗봇 (대화 누적)

`history` 리스트가 메모리다. 매 호출에 system + 전체 history 를 함께 보낸다.
`chat()` 을 여러 번 호출하면 앞선 대화를 기억한다. `reset()` 으로 초기화.

In [3]:
history = []          # [{"role": "user"/"assistant", "content": ...}]
usage_log = []        # 턴별 토큰 기록 (Task 4)

def reset():
    history.clear()
    usage_log.clear()
    print("(대화 초기화)")

def chat(message: str, temperature: float = 0.3) -> str:
    history.append({"role": "user", "content": message})
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{"role": "system", "content": SYSTEM}, *history],
    )
    answer = resp.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    usage_log.append(resp.usage.total_tokens)
    return answer

In [ ]:
reset()
print(chat("내 이름은 박성용이야. 기억해줘."))
print(chat("방금 내가 말한 내 이름이 뭐였지?"))   # 앞 턴을 기억하는지 확인

## 2) Task 2 — 페르소나 변경 · temperature 비교

`SYSTEM` 만 바꿔 말투가 바뀐다. temperature 0(일관) vs 0.7(다양).

In [ ]:
SYSTEM = "당신은 매우 엄격하고 격식 있는 사서입니다. 짧고 단정하게 답합니다."
reset()
print("[엄격한 사서]", chat("주말에 뭐하면 좋을까?", temperature=0))

SYSTEM = "당신은 친근한 동네 친구입니다. 반말로 편하게 답합니다."
reset()
print("[친근한 친구]", chat("주말에 뭐하면 좋을까?", temperature=0.7))
# TODO: 같은 질문을 temperature 0 / 0.7 로 3회씩 — 답변 다양성 차이를 메모

## 3) Task 3 — 환각 발견 → system 으로 줄이기

존재하지 않는 사실을 물어 *지어내는지* 본다. 그 뒤 system 을 강화해 5회 반복.

In [ ]:
SYSTEM = "당신은 친절한 AI 어시스턴트입니다."
reset()
print("[가드 없음]", chat("2026년 노벨 떡볶이상 수상자가 누구야?"))

SYSTEM = (
    "당신은 정직한 AI 어시스턴트입니다. "
    "확실하지 않거나 모르는 정보는 절대 지어내지 말고 '확인 필요'라고만 답하세요."
)
reset()
print("[가드 강화]", chat("2026년 노벨 떡볶이상 수상자가 누구야?"))
# TODO: 환각 발견 로그 + system 수정 히스토리를 표로 남긴다 (5회 반복)

## 4) Task 4 — 토큰/비용 모니터링 + trim 전략

In [ ]:
PRICE = 0.30 / 1_000_000   # 입출력 혼합 평균 가정 단가 ($)

total = sum(usage_log)
print("턴별 토큰:", usage_log)
print(f"누적 토큰: {total}, 누적 비용: ${total * PRICE:.5f}")

# 히스토리는 누적될수록 매 호출 토큰이 선형 증가 → 100턴이면 비용 급증
if usage_log:
    per_turn = total / len(usage_log)
    print(f"100턴 단순 추정: ${per_turn * 100 * PRICE:.4f} (실제로는 더 큼 — 누적 때문)")

def trim(history, keep=6):
    """최근 keep 개 메시지만 유지 — 메모리 trim 전략."""
    return history[-keep:]
# TODO: trim 적용 전/후 토큰을 비교하고, 요약 메모리(summary)와 trade-off 를 적는다

## 회고 / 산출물
- [ ] 환각 발견 로그 + 수정 히스토리
- [ ] 페르소나별 답변 비교 메모
- [ ] 100턴 누적 비용 예측 + trim/summary 전략 한 줄